The dataset can be found -> <a href='https://www.kaggle.com/datasets/yash9439/loan-prediction/data'>Here</a>

In [ ]:
%pip install requirements.txt

In [ ]:
import os
import zipfile
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE
import numpy as np

In [ ]:
cw = os.getcwd()
models = cw+'/models'
extracted_data = cw+'/extracted_data'
zip_path = cw+'/archive.zip'

folders = [models, extracted_data]

if not os.path.exists(extracted_data) or not os.path.exists(models):
    for folder in folders:
        os.makedirs(folder, exist_ok=True)

try:
    with zipfile.ZipFile(zip_path, mode='r') as file:
        file.extractall(path=extracted_data)
except zipfile.BadZipFile as e:
    print(f"Error extracting zipfile: {e}")

In [ ]:
df = pd.read_csv('./extracted_data/Loan_Train.txt')

In [ ]:
df.dropna(inplace=True)
df.isna().sum()

In [ ]:
le = LabelEncoder()
df["Loan_Status"] = le.fit_transform(df["Loan_Status"])
df["Married"] = le.fit_transform(df["Married"])
df["Education"] = le.fit_transform(df["Education"])
df["Self_Employed"] = le.fit_transform(df["Self_Employed"])
df["Gender"] = le.fit_transform(df["Gender"])

df = pd.get_dummies(data=df, columns=["Property_Area"], drop_first=True)


In [ ]:
df["Dependents"] = df["Dependents"].apply(lambda x: 3 if x == '3+' else x)
df["Monthly_Payments"] = df["LoanAmount"] / df["Loan_Amount_Term"]
df["TotalIncome"] = df["ApplicantIncome"] + df["CoapplicantIncome"]
df["Debt_to_Income"] = df["LoanAmount"] / (df["TotalIncome"] + 1)
df["LoanAmount_to_Income"] = df["LoanAmount"] / (df["ApplicantIncome"] + 1)

In [ ]:
df.head()

In [ ]:
y = df["Loan_Status"]
X = df.drop(columns=['Loan_ID', 'Loan_Status', 'ApplicantIncome', 'CoapplicantIncome'])
bool_cols = ['Property_Area_Semiurban', 'Property_Area_Urban']
X[bool_cols] = X[bool_cols].astype(int)

y.value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression()),
])

param_grid = [
    {
        'clf': [LogisticRegression()],
        'clf__max_iter': [100, 500, 1000],
        'clf__solver': ['lbfgs', 'liblinear'],
        'clf__C': [0.1, 1, 10, 30],
        'clf__class_weight': ['balanced', {0:2, 1:1}, {0:3, 1:1}]
    },
    {
        'clf': [SVC(random_state=42, probability=True)],
        'clf__kernel': ['rbf', 'linear'],
        'clf__gamma': ['scale', 0.1, 0.01],
        'clf__C': [0.1, 1, 10, 20],
        'clf__class_weight': ['balanced', {0:2, 1:1}, {0:3, 1:1}]
    },
    {
        'clf': [RandomForestClassifier(random_state=42)],
        'clf__max_depth': [3, 5, 15],
        'clf__n_estimators': [100, 200],
        'clf__min_samples_leaf': [1, 5, 10],
        'clf__class_weight': ['balanced', {0:2, 1:1}, {0:3, 1:1}]
    },
    {
    'clf': [XGBClassifier(random_state=42, eval_metric='logloss')],
    'clf__max_depth': [3, 5, 7],
    'clf__n_estimators': [100, 200],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__scale_pos_weight': [1, 2, 3]
    }
]

grid_search = GridSearchCV(
    pipe, param_grid,
    cv=5,
    scoring='f1_macro'
)

In [ ]:
grid_search.fit(X_train_resampled, y_train_resampled)
print('Best model: ', grid_search.best_estimator_)
print('Best params: ', grid_search.best_params_)
print('Best score: ', grid_search.best_score_)

In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)
ordered_results = results_df.sort_values('rank_test_score', ascending=True)
ordered_results.head(3)

In [ ]:
grid_search.best_params_

In [ ]:
model = RandomForestClassifier(
    class_weight={0: 3, 1: 1},
    max_depth=15,
    min_samples_leaf=1,
    n_estimators=200,
    random_state=42
)
model.fit(X_train_resampled, y_train_resampled)

In [ ]:
proba = model.predict_proba(X_test)

threshold_params = np.arange(0.1, 0.9, 0.05)

scores = [f1_score(y_test, (proba[:, 1] >= t).astype(int), average='macro') 
          for t in threshold_params]

best_threshold = threshold_params[np.argmax(scores)]
print(f"Best Threshold: {best_threshold}\nScore: {max(scores):.3f}")

preds = (proba[:, 1] >= best_threshold).astype(int)

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False))

In [ ]:
print(classification_report(y_test, preds))

In [ ]:
print(confusion_matrix(y_test, preds))

In [ ]:
%pip freeze > requirements.txt